# 20% 情感模型训练（5090 稳妥版）

按顺序逐格运行即可。

## 云 GPU 使用说明（先做这一步）

终端里先准备好代码与环境：

```bash
git clone <你的仓库URL>
cd RSA
python3 -m venv .venv
source .venv/bin/activate
python -m pip install -r ml/requirements-finetune.txt
```

**数据二选一上传：**

- **A. 只训练 20%（推荐省事）**  
  上传整个目录到：  
  `ml/sentiment/data/splits_20pct/`（内含 `train.csv`、`val.csv`、`test.csv`，且列里要有 `analysis_input_en`、`label_sentiment`）  
  → 后面「清洗 / 全量切分 / 再抽 20%」几步会自动跳过。

- **B. 从原始全量重做**  
  上传：`ml/sentiment/data/raw/reviews.csv`  
  → 会依次清洗、全量 8:1:1 划分、再生成 `splits_20pct`。

防断连（可选）：`tmux new -s train`，训练中 `Ctrl+B` 再按 `D` detach；恢复：`tmux attach -t train`。

In [ ]:
from pathlib import Path
import subprocess
import json

ROOT = Path.cwd()
if not (ROOT / "ml").exists():
    raise RuntimeError("请在仓库根目录打开并运行此 Notebook")

PYTHON_BIN = Path(".venv/bin/python")
if not PYTHON_BIN.exists():
    raise RuntimeError("未找到 .venv/bin/python，请先创建虚拟环境并安装依赖")

DIR_20 = ROOT / "ml/sentiment/data/splits_20pct"
PREBUILT_20PCT = all((DIR_20 / name).is_file() for name in ("train.csv", "val.csv", "test.csv"))

RAW_DATA = ROOT / "ml/sentiment/data/raw/reviews.csv"
if not PREBUILT_20PCT and not RAW_DATA.exists():
    raise RuntimeError(
        "数据未就绪：请二选一\n"
        "  A) 上传 ml/sentiment/data/splits_20pct/{train,val,test}.csv\n"
        "  B) 上传 ml/sentiment/data/raw/reviews.csv 并从全流程生成"
    )

CFG_SAFE = ROOT / "ml/sentiment/configs/train_roberta_20pct_gpu_safe.yaml"
if not CFG_SAFE.exists():
    raise RuntimeError("未找到稳妥版配置: ml/sentiment/configs/train_roberta_20pct_gpu_safe.yaml")

print("ROOT:", ROOT)
print("PYTHON_BIN:", PYTHON_BIN)
print("CFG_SAFE:", CFG_SAFE)
print("PREBUILT_20PCT (仅用 splits_20pct):", PREBUILT_20PCT)
if PREBUILT_20PCT:
    print("将跳过：清洗 / 全量切分 / 再抽 20%")
else:
    print("RAW_DATA:", RAW_DATA)

In [ ]:
def run(cmd: list[str]):
    print("\n$", " ".join(cmd))
    p = subprocess.run(cmd, cwd=ROOT)
    if p.returncode != 0:
        raise RuntimeError(f"命令失败: {' '.join(cmd)}")

## 0) GPU 环境自检（推荐）

确认云端已识别 GPU，并检查 PyTorch CUDA 可用性。

In [ ]:
run(["bash", "-lc", "nvidia-smi || true"])

run([
    str(PYTHON_BIN),
    "-c",
    "import torch; print('torch', torch.__version__); print('cuda_available', torch.cuda.is_available()); print('device_count', torch.cuda.device_count())",
])

## 1)~3) 数据准备（仅当从 `raw/reviews.csv` 全流程时需要）

若你已上传 `splits_20pct` 三份 CSV，下面几步会打印「跳过」并直接进入训练。

### 1) 清洗 `raw/reviews.csv`（下一格代码）

In [ ]:
if not PREBUILT_20PCT:
    run([
        str(PYTHON_BIN),
        "ml/sentiment/scripts/clean_data.py",
        "--config", "ml/sentiment/configs/data_contract.yaml",
    ])
else:
    print("跳过：清洗（已上传 splits_20pct）")

### 2) 全量切分 → `ml/sentiment/data/splits/`（下一格代码）

In [ ]:
if not PREBUILT_20PCT:
    run([
        str(PYTHON_BIN),
        "ml/sentiment/scripts/split_dataset.py",
        "--config", "ml/sentiment/configs/train_roberta.yaml",
    ])
else:
    print("跳过：全量切分（已上传 splits_20pct）")

### 行数检查（可选，下一格 `wc`）

In [ ]:
if PREBUILT_20PCT:
    run([
        "bash", "-lc",
        "wc -l ml/sentiment/data/splits_20pct/train.csv ml/sentiment/data/splits_20pct/val.csv ml/sentiment/data/splits_20pct/test.csv",
    ])
else:
    run([
        "bash", "-lc",
        "wc -l ml/sentiment/data/splits/train.csv ml/sentiment/data/splits/val.csv ml/sentiment/data/splits/test.csv",
    ])

### 3) 从 `splits/` 再抽 20% → `splits_20pct`（下一格代码；若已上传 `splits_20pct` 会跳过）

In [ ]:
if not PREBUILT_20PCT:
    run([
        str(PYTHON_BIN),
        "ml/sentiment/scripts/subset_splits.py",
        "--input-dir", "ml/sentiment/data/splits",
        "--output-dir", "ml/sentiment/data/splits_20pct",
        "--fraction", "0.2",
        "--seed", "42",
    ])
else:
    print("跳过：从全量 splits 再抽 20%（你上传的已是 splits_20pct）")

## 4) 开始训练（稳妥版）

In [ ]:
run([
    str(PYTHON_BIN),
    "ml/sentiment/scripts/train_sentiment.py",
    "--config", "ml/sentiment/configs/train_roberta_20pct_gpu_safe.yaml",
    "--tokenized-cache-dir", "ml/sentiment/data/tokenized_cache_20pct",
])

## 5) 查看结果位置

In [ ]:
model_dir = ROOT / "models/sentiment/roberta-sentiment-20pct-gpu5090-safe-v0"
report_path = ROOT / "ml/sentiment/reports/sentiment_eval_20pct_gpu5090_safe_v0.json"

print("模型目录:", model_dir)
print("存在:", model_dir.exists())
print("评估报告:", report_path)
print("存在:", report_path.exists())

if report_path.exists():
    metrics = json.loads(report_path.read_text(encoding="utf-8"))
    print("\n关键指标:")
    for k in ("eval_loss", "eval_accuracy", "eval_runtime", "eval_samples_per_second"):
        if k in metrics:
            print(f"- {k}: {metrics[k]}")
    print("\n完整 JSON:")
    print(json.dumps(metrics, ensure_ascii=False, indent=2))